# scorebook — first pass

**Findings go here, at the top, in plain English.** Not code comments — conclusions a
reader can understand without scrolling. Five sentences, one per question. Write them
last, put them first.

1. Scoring does spike at the death — 9.78 runs per over in the last five against 7.94 in
   the middle — but the powerplay ramps *faster* per over than the death does, so half
   the hypothesis was wrong.
2. Scoring has risen 19% across the 19 seasons, from 8.24 to 9.80 runs per over, and
   almost all of it arrived after 2021 rather than accumulating steadily.
3. A first-over wicket costs the first innings 12.5 runs on average — more than the
   "under 10" predicted, though 10 sits inside the interval.
4. The largest home advantage is Hyderabad's Rajiv Gandhi International Stadium at +19.5
   points of win rate, but league-wide home advantage has shrunk from +8.5 points before
   2016 to +2.6 points after it.
5. Wides are getting *more* common, not less — 4.42 to 5.20 per 100 balls — which
   falsifies Q5 outright. No-balls did fall.

---

The five questions were committed to [`docs/questions.md`](../docs/questions.md) **before**
any analysis, along with what would falsify each one. Read that file before this one.

Loading and cleaning are already done by the package — see
[ADR 0006](../docs/decisions/0006-analysis-in-notebooks.md) for why the analysis is not.

In [ ]:
%matplotlib inline

from pathlib import Path

import pandas as pd

from scorebook import clean, describe, plots
from scorebook.data import loaders

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

# save_fig defaults to `reports/` relative to the working directory, which from here is
# notebooks/reports/. Every chart below passes this instead.
REPORTS = Path("../reports")

In [ ]:
# All 19 seasons. Downloads 6.8 MB the first time, then reads from ~/.cache/scorebook.
#
# Working offline? Swap in the committed sample instead:
#     raw = loaders.load_sample(Path("../data/sample_deliveries.csv"))
raw = loaders.load_deliveries()
deliveries = clean.prepare(raw)

print(describe.format_summary(describe.summarise(deliveries)))
deliveries.head()

## Before anything else: look at the nulls

A high null rate here almost always means "this event is rare", not "this data is
missing". `wicket_type` is 95% null because 95% of deliveries take no wicket. Reaching
for `dropna()` on this frame would delete nearly all of it.

See [ADR 0003](../docs/decisions/0003-informative-nulls.md).

In [ ]:
describe.null_profile(deliveries)

## One thing to know before you group anything

The text columns load as `category`, which cuts memory 7.4×. The cost is that a category
column keeps its full value list after a filter, and `groupby` defaults to
`observed=False` — so it emits one row per *category*, not per value present.

Filter to 2026 and group by team and you get 15 rows for the 10 teams that played, with
five defunct franchises credited with 0 runs. Run the cell below to see it.

**Always pass `observed=True`**, or call `clean.drop_unused_categories(frame)` after
filtering. See [README trap 4](../README.md) and
[the data dictionary](../docs/data-dictionary.md).

In [ ]:
recent = deliveries[deliveries.season_year == 2026]

played = recent["batting_team"].nunique()
unobserved = len(recent.groupby("batting_team", observed=False).size())
observed = len(recent.groupby("batting_team", observed=True).size())

print(f"teams that actually played in 2026: {played}")
print(f"rows from groupby(observed=False):  {unobserved}")
print(f"rows from groupby(observed=True):   {observed}")

# The phantom entries, credited with runs they never scored:
phantom = recent.groupby("batting_team", observed=False)["runs_off_bat"].sum()
phantom[phantom == 0]

## The frame the five questions share

Two decisions, made once here rather than five times below.

**Super overs are dropped.** `innings` above 2 is a tie-breaking shootout, not an innings
— 175 deliveries across the whole archive. Left in, they add sixth innings to matches and
their own over 1 to the death-over profile.

**`total` is `runs_off_bat + extras`.** `runs_off_bat` alone undercounts every delivery
that conceded a wide, a no-ball, a bye or a leg-bye.

In [ ]:
main = clean.drop_unused_categories(deliveries[deliveries.innings <= 2])
main = main.assign(total=main.runs_off_bat + main.extras)

innings_index = main[["match_id", "innings"]].drop_duplicates()

print(f"deliveries          {len(deliveries):>8,}")
print(f"super-over rows     {len(deliveries) - len(main):>8,}  (dropped)")
print(f"analysed            {len(main):>8,}")
print(f"innings             {len(innings_index):>8,}")
print(f"matches             {main.match_id.nunique():>8,}")

## Q1 — Do runs per over spike in the death overs?

**Hypothesis:** yes, and the rise is steeper than the powerplay's.
**Falsified if:** the curve is flat after over 6, or the powerplay is the higher peak.

The total for a delivery is `runs_off_bat + extras` — `runs_off_bat` alone undercounts.
Note that a chase behaves differently from a first innings; consider splitting by
`innings`.

A *Manhattan* is the conventional chart here ([glossary](../docs/glossary.md)).

In [ ]:
# The denominator is the whole question.
#
# Dividing by every innings assumes every innings reaches every over. They do not: a
# chase stops the moment the target is passed, and an innings stops when the tenth
# wicket falls. Only the innings that actually got to an over belong in its denominator.
overs_bowled = main[["match_id", "innings", "over"]].drop_duplicates()
reaching = overs_bowled.groupby("over", observed=True).size()
runs_by_over = main.groupby("over", observed=True)["total"].sum()

rate = runs_by_over / reaching
naive = runs_by_over / len(innings_index)

# `over` is zero-indexed (clean.add_over), so the powerplay is 0-5 and the death is
# 15-19. Where the death overs begin is an argument, not a fact — this is the
# conventional last five, and the chart below lets you disagree with it.
POWERPLAY, MIDDLE, DEATH = range(0, 6), range(6, 15), range(15, 20)


def phase_rate(overs: range) -> float:
    picked = list(overs)
    return runs_by_over[picked].sum() / reaching[picked].sum()


print(f"powerplay (overs 1-6)   {phase_rate(POWERPLAY):.2f} runs/over")
print(f"middle    (overs 7-15)  {phase_rate(MIDDLE):.2f}")
print(f"death     (overs 16-20) {phase_rate(DEATH):.2f}")
print(f"death premium over the middle: {phase_rate(DEATH) - phase_rate(MIDDLE):+.2f}")
print()
print(f"powerplay ramp {(rate[5] - rate[0]) / 5:+.3f} runs/over per over")
print(f"death ramp     {(rate[19] - rate[15]) / 4:+.3f}")
print()
print(f"innings never reaching the last over: {len(innings_index) - reaching[19]:,} "
      f"({100 * (1 - reaching[19] / len(innings_index)):.1f}%)")
print(f"last over, naive denominator {naive[19]:.2f} — correct {rate[19]:.2f}")

# A part-bowled last over still counts as a whole over above, which understates it. Per
# six legal balls instead — wides and no-balls are re-bowled, so they are not deliveries:
legal = main[(main.wides == 0) & (main.noballs == 0)]
per_six = runs_by_over / legal.groupby("over", observed=True).size() * 6
print(f"last over, per six legal balls {per_six[19]:.2f}")

In [ ]:
# Overs are numbered 1-20 for a reader; `over` is 0-indexed underneath.
figure, axes = plots.new_figure(
    "Runs per over, IPL 2008-2026", "over", "runs per over"
)
axes.bar(rate.index + 1, rate.to_numpy(), color="#4c72b0",
         label="denominator: innings that reached the over")
axes.plot(naive.index + 1, naive.to_numpy(), color="#c44e52", marker="o",
          markersize=3.5, linewidth=1.3, label="denominator: every innings (wrong)")
axes.axvspan(0.5, 6.5, color="#dddddd", alpha=0.35, zorder=0)
axes.axvspan(15.5, 20.5, color="#dddddd", alpha=0.35, zorder=0)
axes.set_xticks(range(1, 21))
axes.legend(loc="upper left", fontsize=8)
plots.save_fig(figure, "q1_runs_per_over", reports_dir=REPORTS)

**Finding:** the death overs average **9.78 runs per over** against **7.94** in the middle
overs, a premium of **+1.84** — but the powerplay ramps at **+0.47 runs/over per over**
against the death's **+0.40**, so the rise is *not* steeper than the powerplay's. Half the
hypothesis holds; the half that named the powerplay as the gentler climb does not.

**Caveat:** this pools both innings, and a chase behaves differently from a first innings
once the target is close. It also treats the last over as a whole over even when an
innings ended three balls into it, which understates over 20 slightly — per six legal
balls it is 11.43 rather than 10.59.

## Q2 — Has scoring inflated across 19 seasons?

**Hypothesis:** risen, but less than commentary implies, and unevenly.
**Falsified if:** flat, or non-monotonic in a way no rule change explains.

Use `season_year`, not `season` — the label is a string and three values carry a slash
that no parsing rule handles correctly
([ADR 0004](../docs/decisions/0004-season-year.md)). Remember 2009 was played in South
Africa and 2020 in the UAE.

In [ ]:
# Same denominator problem as Q1, so the same fix: overs actually bowled, not overs
# assumed. A season with more rain-shortened innings would otherwise look low-scoring.
season_of = main[["match_id", "innings", "season_year"]].drop_duplicates()
overs_per_season = (
    overs_bowled.merge(season_of, on=["match_id", "innings"])
    .groupby("season_year", observed=True)
    .size()
)
runs_per_season = main.groupby("season_year", observed=True)["total"].sum()

season = pd.DataFrame({
    "runs": runs_per_season,
    "overs": overs_per_season,
    "rate": runs_per_season / overs_per_season,
    "matches": main.groupby("season_year", observed=True)["match_id"].nunique(),
})

first, last = season.rate.iloc[0], season.rate.iloc[-1]
print(season.round(2).to_string())
print()
print(f"{season.index[0]} {first:.2f} -> {season.index[-1]} {last:.2f} runs/over "
      f"({last - first:+.2f}, {100 * (last / first - 1):+.1f}%)")
print(f"lowest {season.rate.min():.2f} ({season.rate.idxmin()}), "
      f"highest {season.rate.max():.2f} ({season.rate.idxmax()})")
print(f"monotonic? {season.rate.is_monotonic_increasing}  "
      f"biggest fall {season.rate.diff().min():.2f} in {season.rate.diff().idxmin()}")
print(f"mean 2008-2021 {season.rate.loc[:2021].mean():.2f}  "
      f"mean 2022-2026 {season.rate.loc[2022:].mean():.2f}")

In [ ]:
figure, axes = plots.new_figure(
    "Runs per over by season", "season", "runs per over"
)
axes.plot(season.index, season.rate.to_numpy(), marker="o", color="#4c72b0")
# Footroom, so the annotation below the lowest point is not clipped by the axis.
axes.set_ylim(season.rate.min() - 0.35, season.rate.max() + 0.15)
# The two seasons played abroad, which is the readiest explanation for their dips.
for year, label in ((2009, "South Africa"), (2020, "UAE")):
    axes.annotate(label, (year, season.rate[year]), textcoords="offset points",
                  xytext=(0, -16), ha="center", fontsize=8, color="#555555")
axes.set_xticks(season.index[::2])
plots.save_fig(figure, "q2_runs_per_over_by_season", reports_dir=REPORTS)

**Finding:** scoring has risen from **8.24 to 9.80 runs per over**, **+1.56 (+18.9%)**
across the 19 seasons — and it is emphatically uneven. The first fourteen seasons average
8.07 and never exceed 8.59; the last five average 9.25 and climb every year. Almost the
entire increase arrives after 2021.

**Caveat:** not falsified, but "less than commentary implies" looks wrong — a 19% rise
concentrated in four seasons is a large move, not a modest one. The series is
non-monotonic, with the biggest fall (−0.81) in 2009, the season played in South Africa;
the 2022 step up coincides with expansion to ten teams and the 2023 one with the impact
player rule, so the shape has candidate explanations but this data cannot test them.

## Q3 — Does a first-over wicket reduce the innings total?

**Hypothesis:** yes, but by under 10 runs on average.
**Falsified if:** the difference is under 2 runs, or reverses.

The hardest of the five, and the most transferable. "First-over wicket" is not a column:
group to innings level, flag whether any non-null `wicket_type` occurs where `over == 0`,
then join that flag back to compare totals.

**The trap:** innings that ended early — rain, or a chase completed — have low totals for
reasons unrelated to the wicket. Filter to innings of at least 19 overs first, and say so.

In [ ]:
innings_runs = (
    main.groupby(["match_id", "innings"], observed=True)
    .agg(runs=("total", "sum"), balls=("total", "size"))
    .reset_index()
)

# notna() rather than != "none": no wicket is a null, not a dismissal type (ADR 0003).
lost_early = (
    main[(main.over == 0) & main.wicket_type.notna()][["match_id", "innings"]]
    .drop_duplicates()
    .assign(first_over_wicket=True)
)
innings_runs = innings_runs.merge(lost_early, on=["match_id", "innings"], how="left")
innings_runs["first_over_wicket"] = innings_runs.first_over_wicket.notna()

# First innings only. A chase is stopped by the result — 38% of second innings end before
# the last over — so its total measures the target, not what the batting side could make.
first_innings = innings_runs[innings_runs.innings == 1]

summary = first_innings.groupby("first_over_wicket")["runs"].agg(
    ["count", "mean", "median", "std"]
)
gap = summary.loc[True, "mean"] - summary.loc[False, "mean"]
standard_error = (
    summary.loc[True, "std"] ** 2 / summary.loc[True, "count"]
    + summary.loc[False, "std"] ** 2 / summary.loc[False, "count"]
) ** 0.5

print(summary.round(2).to_string())
print()
print(f"difference in means {gap:+.2f} runs")
print(f"~95% interval {gap - 1.96 * standard_error:+.1f} to {gap + 1.96 * standard_error:+.1f}")

In [ ]:
# The guard this notebook suggested — drop innings under 19 overs — is the wrong guard
# for a first innings. A first innings only ends early by being bowled out or rained off,
# and being bowled out is part of the effect being measured. Sure enough, short first
# innings are disproportionately the ones that lost a wicket in the first over, so the
# filter removes exactly the collapses the question is about.
short = first_innings[first_innings.balls < 114]
print(f"first innings under 19 overs: {len(short)} ({100 * len(short) / len(first_innings):.1f}%)")
print(f"  of those, lost a first-over wicket: {short.first_over_wicket.mean():.1%}")
print(f"  base rate across all first innings: {first_innings.first_over_wicket.mean():.1%}")

full_length = first_innings[first_innings.balls >= 114]
guarded = full_length.groupby("first_over_wicket")["runs"].mean()
print()
print(f"with the filter applied anyway: {guarded[True] - guarded[False]:+.2f} runs")
print("— the estimate barely moves, so the conclusion does not rest on the choice.")

In [ ]:
figure, axes = plots.new_figure(
    "First-innings totals, with and without a first-over wicket",
    "innings total",
    "innings",
)
for flag, colour, label in (
    (False, "#4c72b0", f"no first-over wicket (n={summary.loc[False, 'count']:,})"),
    (True, "#c44e52", f"first-over wicket (n={summary.loc[True, 'count']:,})"),
):
    values = first_innings[first_innings.first_over_wicket == flag]["runs"]
    axes.hist(values, bins=range(40, 300, 10), alpha=0.55, color=colour,
              density=True, label=label)
    axes.axvline(values.mean(), color=colour, linestyle="--", linewidth=1.4)
axes.set_ylabel("share of innings")
axes.legend(fontsize=8)
plots.save_fig(figure, "q3_first_over_wicket", reports_dir=REPORTS)

**Finding:** a first-over wicket costs the first innings **12.5 runs** on average —
**158.1** against **170.6** — across 199 innings that lost one and 1,044 that did not.
The rough 95% interval is **−17.5 to −7.5 runs**.

**Caveat:** the hypothesis said "under 10 runs" and the point estimate is 12.5, but 10
sits inside the interval, so this is a mis-estimate rather than a refutation. It is also
an association, not a cost: the same conditions that produce a first-over wicket — a
seaming pitch, a new ball doing something — go on suppressing runs all innings, and
nothing here separates the wicket from the conditions that caused it.

## Q4 — Which venue shows the largest home advantage?

**Hypothesis:** a real but small effect, concentrated in two or three pitches.
**Falsified if:** no venue's advantage survives separating eras.

Match winners live in the 1,243 `_info.csv` files, which are key-value long format. They
are now read by `loaders.load_match_info()` — see
[ADR 0007](../docs/decisions/0007-reading-the-info-files.md).

Two things "home" does not survive: CSK and RR were suspended in 2016–17, and the 2009 and
2020 seasons were played abroad, where nobody was at home.

In [ ]:
# Both frames need canonicalising, and for the same reason in two places: one franchise
# per name, one ground per name.
info = loaders.load_match_info()
info = clean.canonical_teams(info, columns=clean.INFO_TEAM_COLUMNS)
info = clean.canonical_venues(info)
info = info.assign(year=info.start_date.dt.year)

# One row per team per match, so a match contributes to both sides' records.
appearances = pd.concat([
    info[["match_id", "venue", "year", "winner", side]].rename(columns={side: "team"})
    for side in ("team_1", "team_2")
])
for column in ("team", "venue", "winner"):
    appearances[column] = appearances[column].astype("string")

# 2009 (South Africa) and 2020 (UAE) are dropped before "home" is defined, not after —
# otherwise a season abroad votes on where a team's home ground is.
ABROAD = {2009, 2020}
domestic = appearances[~appearances.year.isin(ABROAD)]

# "Home" is a choice, and this is it: the ground a team has played most often. It is not
# in the data and it is not in the package for that reason.
home_ground = (
    domestic.groupby(["team", "venue"], observed=True).size().rename("played").reset_index()
    .sort_values(["team", "played"], ascending=[True, False])
    .drop_duplicates("team")
)

decided = domestic[domestic.winner.notna()].merge(
    home_ground[["team", "venue"]].rename(columns={"venue": "home"}), on="team", how="left"
)
decided = decided.assign(
    at_home=decided.venue == decided.home, won=decided.winner == decided.team
)

print(f"matches {len(info):,}   decided {int(info.winner.notna().sum()):,}   "
      f"no winner {int(info.winner.isna().sum())} "
      f"({int((info.outcome == 'tie').sum())} ties, "
      f"{int((info.outcome == 'no result').sum())} no results)")
print(f"grounds after canonicalising: {info.venue.nunique()}")

In [ ]:
rates = decided.groupby(["team", "at_home"], observed=True)["won"].agg(["size", "mean"]).unstack()
rates.columns = ["away_n", "home_n", "away_rate", "home_rate"]
rates["advantage"] = rates.home_rate - rates.away_rate
rates = (
    rates.join(home_ground.set_index("team")["venue"])
    .dropna()
    .sort_values("advantage", ascending=False)
)

print(rates[["venue", "home_n", "home_rate", "away_n", "away_rate", "advantage"]]
      .round(3).to_string())
print()
print(f"league-wide home {decided[decided.at_home].won.mean():.3f} "
      f"(n={int(decided.at_home.sum()):,})   "
      f"away {decided[~decided.at_home].won.mean():.3f} "
      f"(n={int((~decided.at_home).sum()):,})")

In [ ]:
# The falsification criterion, tested rather than asserted.
decided = decided.assign(
    era=pd.cut(decided.year, [2007, 2015, 2026], labels=["2008-2015", "2016-2026"])
)
print(decided.groupby(["era", "at_home"], observed=True)["won"]
      .agg(["size", "mean"]).round(3).to_string())

MIN_MATCHES = 15
by_era = []
for (team, era), group in decided.groupby(["team", "era"], observed=True):
    home, away = group[group.at_home], group[~group.at_home]
    if len(home) >= MIN_MATCHES and len(away) >= MIN_MATCHES:
        by_era.append({"team": team, "era": str(era),
                       "advantage": home.won.mean() - away.won.mean()})

eras = pd.DataFrame(by_era).pivot(index="team", columns="era", values="advantage").dropna()
eras["survives"] = (eras.iloc[:, 0] > 0) & (eras.iloc[:, 1] > 0)
print()
print(eras.round(3).sort_values(eras.columns[0], ascending=False).to_string())
print()
print(f"positive in both eras: {int(eras.survives.sum())} of {len(eras)}")

In [ ]:
# Sample size is in the label because it is the story: no ground has 100 home matches.
LABEL_LIMIT = 34
shown = rates[rates.home_n >= 25]
labels = [
    f"{venue if len(venue) <= LABEL_LIMIT else venue[:LABEL_LIMIT - 1] + '…'}  (n={int(n)})"
    for venue, n in zip(shown.venue, shown.home_n, strict=True)
]

figure, axes = plots.new_figure(
    "Home advantage by ground",
    "home win rate minus away win rate, percentage points",
    "",
)
positions = range(len(shown))
axes.barh(list(positions), (shown.advantage * 100).to_numpy(),
          color=["#4c72b0" if v > 0 else "#c44e52" for v in shown.advantage])
axes.set_yticks(list(positions))
axes.set_yticklabels(labels, fontsize=8)
axes.invert_yaxis()
axes.axvline(0, color="#333333", linewidth=0.8)
# The exclusions belong on the chart, not only in the prose around it.
axes.text(0.99, 0.02, "2009 (South Africa) and 2020 (UAE) excluded",
          transform=axes.transAxes, ha="right", fontsize=7, color="#666666")
plots.save_fig(figure, "q4_home_advantage", reports_dir=REPORTS)

**Finding:** the largest is Hyderabad's **Rajiv Gandhi International Stadium at +19.5
percentage points** — Sunrisers win 61.8% there (n=68) against 42.3% elsewhere (n=123).
Chepauk follows at +14.6 and Sawai Mansingh at +11.4. League-wide the effect is real but
small: **53.3% at home against 48.2% away, +5.1 points**. Nine of the fifteen teams show
no advantage at all.

**Caveat:** the hypothesis survives its falsification test — four of the eight teams with
enough matches in both eras keep a positive advantage — but only just, and the pooled
ranking is mostly a fossil. Home advantage has fallen from **+8.5 points before 2016 to
+2.6 after**, and Rajasthan, the largest advantage in the early era at +26.4, *reverses*
to −3.4 in the late one. Sample sizes are small throughout: the biggest home record here
is 100 matches, most are under 70, and a 5-point difference on 68 matches is roughly one
extra win every three seasons.

Two grounds are shared across the split. Deccan Chargers and Sunrisers Hyderabad both call
the Rajiv Gandhi International Stadium home and sit at opposite ends of the table
(−24.8 and +19.5), which is a reminder that this measures teams at grounds, not grounds.

## Q5 — Are wides and no-balls getting rarer?

**Hypothesis:** slightly rarer, and smaller than the between-season noise.
**Falsified if:** rising, or the variance swamps the trend — in which case "cannot tell
from this data" is the honest answer.

`clean.fill_extras` already turned the nulls into zeros, which matters here: the rate needs
every delivery in its denominator, not only the ones that conceded an extra.

In [ ]:
extras = main.groupby("season_year", observed=True).agg(
    balls=("total", "size"), wides=("wides", "sum"), noballs=("noballs", "sum")
)
extras["wide_rate"] = extras.wides / extras.balls * 100
extras["noball_rate"] = extras.noballs / extras.balls * 100

years = pd.Series(extras.index, index=extras.index).astype(float)
print(extras.round(3).to_string())
print()
for name in ("wide_rate", "noball_rate"):
    series = extras[name]
    print(f"{name:<12} {series.iloc[0]:.2f} -> {series.iloc[-1]:.2f} per 100 balls "
          f"({series.iloc[-1] - series.iloc[0]:+.2f}); "
          f"season-to-season sd {series.std():.3f}; "
          f"correlation with year {series.corr(years):+.3f}")

In [ ]:
figure, axes = plots.new_figure(
    "Wides and no-balls per 100 deliveries", "season", "per 100 deliveries"
)
axes.plot(extras.index, extras.wide_rate.to_numpy(), marker="o", color="#4c72b0",
          label="wides")
axes.plot(extras.index, extras.noball_rate.to_numpy(), marker="o", color="#c44e52",
          label="no-balls")
axes.set_xticks(extras.index[::2])
axes.set_ylim(bottom=0)
axes.legend(fontsize=8)
plots.save_fig(figure, "q5_wides_and_noballs", reports_dir=REPORTS)

**Finding:** **falsified.** Wides are getting *more* common, not rarer — **4.42 to 5.20
per 100 balls**, correlation with season **+0.43**, and 2026 is the highest of the 19
seasons. No-balls did fall, 0.61 to 0.36, correlation **−0.31**. The two move in opposite
directions, so "extras" as a single trend does not exist.

**Caveat:** the second falsification condition also fires. The wide rate's season-to-season
standard deviation is 0.62 against a total drift of 0.78 across nineteen years, so the
noise is nearly the size of the trend and no single season tells you anything. This is a
direction, not a rate of change — and none of it separates umpiring interpretation from
bowler behaviour, which is what the question was really asking.

## What didn't work

The section that makes the rest credible. An aggregation that double-counted, a hypothesis
that died, a confound with no available control.

An empty section here by the end means the questions were too safe — not that everything
worked.

**1. Q1's denominator was wrong, and it inverted the answer.** Dividing runs in an over by
*every* innings makes the last over look like 8.12 runs and *falling* from over 18. It is
10.59 and rising. 578 innings — 23.3% — never reach the last over, because a chase stops
at the target and an innings stops at the tenth wicket. The red line on the Q1 chart is
that mistake, kept deliberately. The glossary already had a word for this — *right-censored*
— and it got written wrong anyway.

**2. Q4 was computed on venue *names* first, and Chennai appeared to abandon their own
ground in 2016.** Cricsheet does not normalise venues: `MA Chidambaram Stadium, Chepauk`
holds 48 matches and `MA Chidambaram Stadium, Chepauk, Chennai` holds 41, split almost
exactly at that year. 60 strings, 36 grounds. Every home sample was a fragment and CSK
dropped out of the era comparison entirely for want of matches. The data dictionary says
"names not normalised upstream" in plain sight. Fixed in `clean.canonical_venues`.

**3. This notebook's own advice on Q3 was wrong.** It said to filter to innings of at least
19 overs. For a first innings that removes the collapses — short first innings are 23.3%
first-over-wicket against a 16.0% base rate — which is exactly the effect being measured.
The estimate survives it (−12.29 against −12.52), so the error changed nothing here, but
it would have on a smaller effect.

**4. Two of the five hypotheses were wrong.** Q5 is falsified outright: wides are rising.
Q1's "steeper than the powerplay" is wrong, though the death-over spike is real. Q2's
"less than commentary implies" looks wrong too, at +18.9%.

**5. Nothing here is causal.** Q3 measures an association between a first-over wicket and a
lower total and cannot separate the wicket from the pitch that produced it. Q4 defines
"home" as a team's modal ground, which is a choice, and cannot separate crowd from pitch
from travel.

**6. `actual_delivery` is still unresolved rather than understood.** It disagrees with
`ball` on 12.05% of rows and nothing above depends on it, which is not the same as it
being harmless — see [ADR 0002](../docs/decisions/0002-ball-not-actual-delivery.md).

---

The findings are copied into [`docs/results.md`](../docs/results.md); the charts are in
`reports/`, regenerated by re-running this notebook rather than committed.